## Importing Packages

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor

import mlflow
import mlflow.sklearn
import mlflow.xgboost

import dagshub

## DagsHub MLflow Setup

In [2]:
dagshub.init(
    repo_owner="Nivedhaasai",
    repo_name="MLflow-Regression",
    mlflow=True
)

Accessing as Nivedhaasai
Initialized MLflow to track repo "Nivedhaasai/MLflow-Regression"
Repository Nivedhaasai/MLflow-Regression initialized!


In [3]:
mlflow.set_experiment(
    "California Housing Regression PBLM 1"
)

2026/08/06 11:36:25 INFO mlflow.tracking.fluent: Experiment with name 'California Housing Regression PBLM 1' does not exist. Creating a new experiment.


## Data Loading and Processing

In [4]:
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target

print("Shape:", X.shape)
print(y.describe())

Shape: (20640, 8)
count    20640.000000
mean         2.068558
std          1.153956
min          0.149990
25%          1.196000
50%          1.797000
75%          2.647250
max          5.000010
Name: MedHouseVal, dtype: float64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Build Models

In [6]:
models = [
    (
        "Linear Regression",
        LinearRegression(),
        X_train,
        y_train
    ),
    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            random_state=42
        ),
        X_train,
        y_train
    ),
    (
        "XGBoost",
        XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            random_state=42
        ),
        X_train,
        y_train
    )
]

## Train + Evaluate (R2 is the metric we care about)

In [7]:
reports = []
trained_models = []
for model_name, model, X_tr, y_tr in models:
    model.fit(
        X_tr,
        y_tr
    )
    predictions = model.predict(
        X_test
    )
    report = {
        "mse": mean_squared_error(y_test, predictions),
        "rmse": float(np.sqrt(mean_squared_error(y_test, predictions))),
        "mae": mean_absolute_error(y_test, predictions),
        "r2": r2_score(y_test, predictions)
    }
    reports.append(report)

    trained_models.append(
        model
    )
    print("="*50)
    print(model_name)
    print("="*50)
    print("R2  :", report["r2"])
    print("RMSE:", report["rmse"])
    print("MAE :", report["mae"])

Linear Regression
R2  : 0.5757877060324514
RMSE: 0.7455813830127758
MAE : 0.5332001304956557
Random Forest
R2  : 0.774774131273305
RMSE: 0.5432660494744426
MAE : 0.3657301973669837
XGBoost
R2  : 0.8408716044998452
RMSE: 0.45664349131075066
MAE : 0.2959191474993146


## Log All Experiments to DagsHub

In [8]:
for i, (model_name, model, _, _) in enumerate(models):
    report = reports[i]

    with mlflow.start_run(
        run_name=model_name
    ):

        mlflow.log_param(
            "Model",
            model_name
        )

        mlflow.log_params(
            model.get_params()
        )

        mlflow.log_metric(
            "R2",
            report["r2"]
        )

        mlflow.log_metric(
            "MSE",
            report["mse"]
        )

        mlflow.log_metric(
            "RMSE",
            report["rmse"]
        )

        mlflow.log_metric(
            "MAE",
            report["mae"]
        )

        if "XGBoost" in model_name:
            mlflow.xgboost.log_model(
                model,
                name="model"
            )
        else:
            mlflow.sklearn.log_model(
                model,
                name="model"
            )

print("Experiments logged to DagsHub!")

🏃 View run Linear Regression at: https://dagshub.com/Nivedhaasai/MLflow-Regression.mlflow/#/experiments/0/runs/d8b26494d39e47dd923380a0e4e936b3
🧪 View experiment at: https://dagshub.com/Nivedhaasai/MLflow-Regression.mlflow/#/experiments/0
🏃 View run Random Forest at: https://dagshub.com/Nivedhaasai/MLflow-Regression.mlflow/#/experiments/0/runs/f0827095e4e64092b80c006936c9b677
🧪 View experiment at: https://dagshub.com/Nivedhaasai/MLflow-Regression.mlflow/#/experiments/0
🏃 View run XGBoost at: https://dagshub.com/Nivedhaasai/MLflow-Regression.mlflow/#/experiments/0/runs/95852a76b65e40e7b3eccdd8250add2b
🧪 View experiment at: https://dagshub.com/Nivedhaasai/MLflow-Regression.mlflow/#/experiments/0
Experiments logged to DagsHub!


## Best Model (by R2) and Register to DagsHub

In [9]:
best_index = np.argmax(
    [
        r["r2"]
        for r in reports
    ]
)
best_model_name = models[best_index][0]

best_model = trained_models[best_index]

best_report = reports[best_index]

print(
    "Best Model:",
    best_model_name
)
print(
    "Best R2:",
    best_report["r2"]
)

Best Model: XGBoost
Best R2: 0.8408716044998452


In [10]:
with mlflow.start_run(
    run_name=f"Champion_{best_model_name}"
) as run:
    mlflow.log_param(
        "Model",
        best_model_name
    )
    mlflow.log_metric(
        "R2",
        best_report["r2"]
    )
    mlflow.log_metric(
        "RMSE",
        best_report["rmse"]
    )
    if "XGBoost" in best_model_name:

        mlflow.xgboost.log_model(
            best_model,
            name="model",
            registered_model_name="California_Housing_Best_Model"
        )
    else:

        mlflow.sklearn.log_model(
            best_model,
            name="model",
            registered_model_name="California_Housing_Best_Model"
        )
    run_id = run.info.run_id

print(run_id)

Successfully registered model 'California_Housing_Best_Model'.
2026/08/06 11:39:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: California_Housing_Best_Model, version 1
Created version '1' of model 'California_Housing_Best_Model'.
🏃 View run Champion_XGBoost at: https://dagshub.com/Nivedhaasai/MLflow-Regression.mlflow/#/experiments/0/runs/6d6bb97d6a9246a2a76f7dbb705272fd
🧪 View experiment at: https://dagshub.com/Nivedhaasai/MLflow-Regression.mlflow/#/experiments/0
6d6bb97d6a9246a2a76f7dbb705272fd
